In [9]:
# import packages and BTMS_model

import os
import io
import contextlib
import itertools
import numpy as np
import pandas as pd

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

import lib.BTMS_model as BTMS_model
import CoolProp.CoolProp as CP

In [10]:
# result-file naming configuration
# User-selected filename convention:
# [RESULTID]_[SCN]_[BTMS]_[MODEL]_[TASK].[ext]
# This notebook is for Park liquid-cooling parameter scanning using the 1D model.

CASE_ID = 'C0003'
REVISION = 'R02'
RESULT_ID = f'{CASE_ID}{REVISION}'

SCENARIO_CODE = 'PARK25'
BTMS_CODE = 'LC'
MODEL_CODE = '1D'
TASK_CODE = 'SCAN'
# This scan varies m_dot, T_water_in, hydraulic diameter Dh, and num_cell_seg.
# The task suffix SCAN is intentionally included here to match the selected
# notebook/result naming: C0003R02_PARK25_LC_1D_SCAN.*
RESULT_BASENAME = f'{RESULT_ID}_{SCENARIO_CODE}_{BTMS_CODE}_{MODEL_CODE}_{TASK_CODE}'
NOTEBOOK_NAME = f'{RESULT_BASENAME}.ipynb'
OUTPUT_DIR = os.path.join(os.getcwd(), 'data_results')

RESULT_XLSX_NAME = f'{RESULT_BASENAME}.xlsx'

print(f'Result basename: {RESULT_BASENAME}')
print(f'Notebook file: {NOTEBOOK_NAME}')
print(f'Official Excel result: {RESULT_XLSX_NAME}')


Result basename: C0003R02_PARK25_LC_1D_SCAN
Notebook file: C0003R02_PARK25_LC_1D_SCAN.ipynb
Official Excel result: C0003R02_PARK25_LC_1D_SCAN.xlsx


In [11]:
# read Qbat(W) from the data file

# Keep the physical simulation duration unchanged, while using the original time step.
SIM_DURATION_S = 1920.0   # s, physical mission duration
dt = 1.0                 # s, integration time step

NUM_STEPS = int(round(SIM_DURATION_S / dt))
if not np.isclose(NUM_STEPS * dt, SIM_DURATION_S):
    raise ValueError('SIM_DURATION_S must be an integer multiple of dt.')

# Keep the original variable name for downstream compatibility.
# Here SIM_TIME_S means the number of numerical time steps, not the physical duration.
SIM_TIME_S = NUM_STEPS

file_name = os.path.join(
    os.getcwd(),
    'data',
    'MD05E070207A1  data_power gen_single cell_Liu_20260421.xlsx'
)

if not os.path.isfile(file_name):
    raise FileNotFoundError(
        f'Heat-generation data file not found: {file_name}\n'
        'Place the Excel file in the data folder beside this notebook.'
    )

df = pd.read_excel(file_name, sheet_name='Sheet1')
power_generation_data_1s = df['Qbat(W)'].dropna().to_numpy(dtype=float)

# The source Qbat data are treated as 1-s samples. For dt = 1.0 s,
# each 1-s heat-generation value is used for one numerical time step.
time_s = np.arange(SIM_TIME_S + 1) * dt
power_indices = np.floor(time_s[:-1]).astype(int)
power_indices = np.clip(power_indices, 0, len(power_generation_data_1s) - 1)
power_generation_data = power_generation_data_1s[power_indices]

print(f'Simulation duration: {SIM_DURATION_S:.1f} s')
print(f'Time step dt: {dt:.3f} s')
print(f'Number of numerical steps: {SIM_TIME_S}')
print(f'Q_gen array length: {len(power_generation_data)}')

Simulation duration: 1920.0 s
Time step dt: 1.000 s
Number of numerical steps: 1920
Q_gen array length: 1920


In [12]:
# model settings and parameter-scan ranges
#
# Maintenance note:
# In most cases, only edit the dictionaries/lists in this cell.
# The original variable names are kept below so that the downstream
# calculation logic and notebook structure do not need to be changed.

# battery cell properties
# Initial battery temperature is set equal to the scanned inlet water temperature in each case.
BATTERY_PROPS = {
    'm_bat': 48e-3,          # kg
    'cp_bat': 830.0,         # J/(kg K)
    'D_bat': 18e-3,          # m
    'H_bat': 65e-3,          # m
}

# 1D module layout
MODULE_LAYOUT = {
    'N_r': 20,               # cells/control volumes along the coolant-flow direction
    'N_c': 16,               # battery columns in the transverse direction
}

# coolant / water settings
# Note: the liquid-cooling working fluid is water. The air-cooling notebook uses air.
# The circular-channel hydraulic diameter is now a scan variable (Dh_list below),
# so no fixed D_channel is stored in this dictionary.
LIQUID_SETTINGS = {
    'fluid': 'Water',
    'p_water': 101325.0,      # Pa
    'L_channel': 0.41423277,  # m, fixed channel length
    'plate_thickness': 0.012, # m, equivalent conduction thickness used in R_plate
    'k_plate': 202.4,         # W/(m K), aluminium thermal conductivity

    # Fixed Park-module and aluminium-plate geometry used for mass calculation
    'S_T': 1.25 * BATTERY_PROPS['D_bat'], # m, transverse cell pitch
    'plate_extra_height': 4e-3,            # m, plate height = Dh + 4 mm
    'rho_plate': 2719.0,                   # kg/m3, aluminium density

    # Pump and additional-component settings
    'pump_efficiency': 0.35,
    'K_minor': 0.0,
    'm_pump': 0.0,                         # kg; update when pump mass is specified
    'm_pipe': 0.0,                         # kg; update when pipe mass is specified
}

# liquid-cooling operating period
# For the liquid-cooling scan, water cooling is kept active during the full mission.
# If the liquid-cooling system needs to follow the same cruise-only logic as the
# air-cooling case, set the corresponding time condition in run_one_case.
COOLING_PERIOD = {
    't_cruise_start': 180.0,
    't_cruise_end': 1560.0,
}

# solver settings passed to BTMS_model.solve_coolant_temperature_distribution
SOLVER_SETTINGS = {
    'solver_tol': 1e-6,
    'solver_maxiter': 1000,
}

# mission-stage definitions for table indicators
mission_stages = [
    ('takeoff',       0.0,    6.0),
    ('climb',         6.0,   36.0),
    ('transition1',  36.0,  180.0),
    ('cruise',      180.0, 1560.0),
    ('transition2',1560.0, 1704.0),
    ('descent',    1704.0, 1734.0),
    ('hover',      1734.0, 1914.0),
    ('landing',    1914.0, 1920.0),
]

# parameter-scan ranges
# Engineering-oriented full-factorial scan:
# - T_water: inlet-water temperature, degC
# - m_dot_total: total mass flow rate of the complete liquid-cooling system, kg/s
# - num_cell_seg: equivalent number of transverse cells represented by each channel
# - Dh: hydraulic diameter of each cooling channel, m

SCAN_VALUES = {
    'T_water_list': np.array([
        20.0,
        25.0,
        30.0,
        35.0,
    ], dtype=float),

    # Total mass flow rate of the complete liquid-cooling system.
    # Unit: kg/s.
    'm_dot_total_list': np.array([
        0.045,
        0.050,
        0.055,
        0.060,  # baseline total mass flow rate
        0.065,
        0.070,
    ], dtype=float),

    'num_cell_seg_list': np.array([
        4.0/7.0,   # 28 parallel channels
        2.0/3.0,   # 24 parallel channels
        2.0/2.5,   # 20 parallel channels
        1.00,      # 16 parallel channels
        2.0/1.5,   # 12 parallel channels
    ], dtype=float),

    # Hydraulic diameter of each circular cooling channel.
    'Dh_list': np.array([
        3.0,
        4.0,
        5.0,
        6.0,
    ], dtype=float) * 1e-3,
}

# Map configuration values to variables used by the downstream calculations.
m_bat = BATTERY_PROPS['m_bat']
cp_bat = BATTERY_PROPS['cp_bat']
D_bat = BATTERY_PROPS['D_bat']
H_bat = BATTERY_PROPS['H_bat']
A_battery = np.pi * D_bat * H_bat

N_r = MODULE_LAYOUT['N_r']
N_c = MODULE_LAYOUT['N_c']
num_seg = N_r

fluid = LIQUID_SETTINGS['fluid']
p_water = LIQUID_SETTINGS['p_water']
L_channel = LIQUID_SETTINGS['L_channel']
plate_thickness = LIQUID_SETTINGS['plate_thickness']
k_plate = LIQUID_SETTINGS['k_plate']
S_T = LIQUID_SETTINGS['S_T']
plate_extra_height = LIQUID_SETTINGS['plate_extra_height']
rho_plate = LIQUID_SETTINGS['rho_plate']
pump_efficiency = LIQUID_SETTINGS['pump_efficiency']
K_minor = LIQUID_SETTINGS['K_minor']
m_pump = LIQUID_SETTINGS['m_pump']
m_pipe = LIQUID_SETTINGS['m_pipe']

t_cruise_start = COOLING_PERIOD['t_cruise_start']
t_cruise_end = COOLING_PERIOD['t_cruise_end']

# Liquid cooling is active during the full mission in this notebook.
pump_operation_time = SIM_DURATION_S

solver_tol = SOLVER_SETTINGS['solver_tol']
solver_maxiter = SOLVER_SETTINGS['solver_maxiter']

T_water_list = SCAN_VALUES['T_water_list']
m_dot_total_list = SCAN_VALUES['m_dot_total_list']
num_cell_seg_list = SCAN_VALUES['num_cell_seg_list']
Dh_list = SCAN_VALUES['Dh_list']


In [13]:
# build parameter-scan cases

def build_scan_cases(
    T_water_list,
    m_dot_total_list,
    num_cell_seg_list,
    Dh_list,
):
    """Build all parameter combinations and assign subcase IDs under CASE_ID."""
    cases = []

    for case_count, (
        T_water,
        m_dot_total,
        num_cell_seg,
        Dh,
    ) in enumerate(
        itertools.product(
            T_water_list,
            m_dot_total_list,
            num_cell_seg_list,
            Dh_list,
        ),
        start=1,
    ):
        cases.append({
            'case_id': f'{CASE_ID}_S{case_count:03d}',
            'T_water': float(T_water),
            'm_dot_total': float(m_dot_total),
            'num_cell_seg': float(num_cell_seg),
            'Dh': float(Dh),
        })

    return cases, pd.DataFrame(cases)


scan_cases, scan_cases_df = build_scan_cases(
    T_water_list,
    m_dot_total_list,
    num_cell_seg_list,
    Dh_list,
)

print(f'Total cases: {len(scan_cases)}')
print(
    'Inlet water temperatures: '
    f'{scan_cases_df["T_water"].drop_duplicates().to_numpy()} degC'
)
print(
    'Total mass flow rates: '
    f'{scan_cases_df["m_dot_total"].drop_duplicates().to_numpy()} kg/s'
)
print(
    'Equivalent cells per channel: '
    f'{scan_cases_df["num_cell_seg"].drop_duplicates().to_numpy()}'
)
print(
    'Hydraulic diameters: '
    f'{scan_cases_df["Dh"].drop_duplicates().to_numpy() * 1e3} mm'
)

scan_cases_df.head()


Total cases: 480
Inlet water temperatures: [20. 25. 30. 35.] degC
Total mass flow rates: [0.045 0.05  0.055 0.06  0.065 0.07 ] kg/s
Equivalent cells per channel: [0.57142857 0.66666667 0.8        1.         1.33333333]
Hydraulic diameters: [3. 4. 5. 6.] mm


,case_id,T_water,m_dot_total,num_cell_seg,Dh
0,C0003_S001,20.0,0.045,0.571429,0.003
1,C0003_S002,20.0,0.045,0.571429,0.004
2,C0003_S003,20.0,0.045,0.571429,0.005
3,C0003_S004,20.0,0.045,0.571429,0.006
4,C0003_S005,20.0,0.045,0.666667,0.003


In [14]:
# calculate water properties once for each inlet water temperature

water_props_cache = {}

for T_water in sorted(scan_cases_df['T_water'].unique()):
    T_water = float(T_water)
    T_water_K = T_water + 273.15

    mu_water = CP.PropsSI('V', 'T', T_water_K, 'P', p_water, fluid)
    rho_water = CP.PropsSI('D', 'T', T_water_K, 'P', p_water, fluid)
    cp_water = CP.PropsSI('C', 'T', T_water_K, 'P', p_water, fluid)
    k_water = CP.PropsSI('L', 'T', T_water_K, 'P', p_water, fluid)
    Pr_water = cp_water * mu_water / k_water

    water_props_cache[T_water] = {
        'mu_water': mu_water,
        'rho_water': rho_water,
        'cp_water': cp_water,
        'k_water': k_water,
        'Pr_water': Pr_water,
    }

print(f'Water properties calculated for {len(water_props_cache)} inlet temperatures.')

Water properties calculated for 4 inlet temperatures.


In [15]:
# run one parameter-scan case by calling BTMS_model

def run_one_case(case_id, T_water, m_dot_total, num_cell_seg, Dh):
    water_props = water_props_cache[float(T_water)]

    if Dh <= 0:
        raise ValueError('Dh must be positive.')

    if m_dot_total <= 0:
        raise ValueError('m_dot_total must be positive.')

    if num_cell_seg <= 0:
        raise ValueError('num_cell_seg must be positive.')

    # ---------------------------------------------------------
    # Number of parallel channels and per-channel mass flow rate
    # ---------------------------------------------------------
    num_parallel_channels_float = N_c / num_cell_seg
    num_parallel_channels = int(round(num_parallel_channels_float))

    if not np.isclose(
        num_parallel_channels,
        num_parallel_channels_float,
    ):
        raise ValueError(
            f'num_cell_seg={num_cell_seg} does not produce '
            'an integer channel count.'
        )

    # The parameter scan uses the total system mass flow rate.
    # The 1D thermal solver uses the mass flow rate through one channel.
    m_dot_channel = m_dot_total / num_parallel_channels

    # ---------------------------------------------------------
    # Case-specific circular-channel geometry and heat transfer
    # For a circular channel, hydraulic diameter Dh equals
    # the physical channel diameter.
    # ---------------------------------------------------------
    A_cool_cs = np.pi * Dh ** 2 / 4
    A_HT_seg = np.pi * Dh * L_channel / num_seg

    u_water = (
        m_dot_channel
        / (water_props['rho_water'] * A_cool_cs)
    )

    Re_water = (
        water_props['rho_water']
        * u_water
        * Dh
        / water_props['mu_water']
    )

    Nu_water = BTMS_model.liquid_nusselt_number(
        Re_water,
        water_props['Pr_water'],
        heating=True,
    )

    h_water = Nu_water * water_props['k_water'] / Dh
    R_plate = plate_thickness / k_plate
    htc_global = 1.0 / (1.0 / h_water + R_plate)
    htc_cool = np.ones(num_seg) * htc_global

    # The pressure drop is calculated using the flow conditions
    # in one representative channel, while the total pump power
    # is calculated using the total system mass flow rate.
    pump_args = {
        'fluid_cool': fluid,
        'rho_cool': water_props['rho_water'],
        'mu_cool': water_props['mu_water'],
        
        'A_cool_cs': A_cool_cs,
        'm_dot_total': m_dot_total,
        'num_channel': num_parallel_channels,

        'D_channel': Dh,
        'L_channel': L_channel,
        'pump_efficiency': pump_efficiency,
        'K_minor': K_minor,
        'operation_time': pump_operation_time,
    }

    pump_results = BTMS_model.cal_btms_aux_power(pump_args)
    P_pump = float(pump_results['P_aux_W'])
    E_pump_cumulative = float(pump_results['E_aux_J'])

    # The Reynolds number used in the pump calculation should be
    # identical to the Reynolds number used for heat transfer.
    if not np.isclose(
        pump_results['Re'],
        Re_water,
        rtol=1e-12,
        atol=0.0,
    ):
        raise RuntimeError(
            'Inconsistent Reynolds number in pump-power calculation.'
        )

    # ---------------------------------------------------------
    # Full liquid-cooling BTMS mass
    # ---------------------------------------------------------
    mass_args = {
        'fluid_cool': fluid,
        'rho_cool': water_props['rho_water'],
        'A_cool_cs': A_cool_cs,
        'L_channel': L_channel,
        'D_channel': Dh,
        'num_channel': num_parallel_channels,
        'D_bat': D_bat,
        'S_T': S_T,
        'N_c': N_c,
        'plate_extra_height': plate_extra_height,
        'rho_plate': rho_plate,
        'm_pump': m_pump,
        'm_pipe': m_pipe,
        'return_components': True,
    }

    mass_results = BTMS_model.cal_btms_mass(mass_args)
    mtotal = float(mass_results['m_BTMS_kg'])

    mass_component_sum = (
        mass_results['m_plate_kg']
        + mass_results['m_coolant_kg']
        + mass_results['m_pump_kg']
        + mass_results['m_pipe_kg']
    )

    if not np.isclose(
        mtotal,
        mass_component_sum,
        rtol=1e-12,
        atol=1e-12,
    ):
        raise RuntimeError(
            'Inconsistent component sum in BTMS mass calculation.'
        )

    # Initial battery and coolant temperatures are equal to
    # the scanned inlet-water temperature.
    T_bat = np.ones(num_seg) * T_water
    T_cool = np.ones(num_seg) * T_water

    T_bat_history = np.zeros((SIM_TIME_S + 1, num_seg))
    T_water_history = np.zeros((SIM_TIME_S + 1, num_seg))

    T_bat_history[0, :] = T_bat
    T_water_history[0, :] = T_cool

    for k in range(SIM_TIME_S):
        t = time_s[k]
        is_cool = True

        # The thermal solver represents one parallel cooling channel.
        # Therefore, it uses the per-channel coolant velocity and the
        # equivalent number of cells represented by that channel.
        args = {
            'num_seg': num_seg,
            'num_seg_bat': num_seg,
            'dt': dt,
            'T_cool_pre': T_cool,
            'T_bat_pre': T_bat,
            'u_cool_in': u_water,
            'p_cool': p_water,
            'fluid_cool': fluid,
            'A_HT_seg': A_HT_seg,
            'A_cool_cs': A_cool_cs,
            'm_bat': m_bat,
            'cp_bat': cp_bat,
            'D_bat': D_bat,
            'T_cool_in': T_water,
            'T_cool_out': max(
                float(T_cool[-1]),
                T_water,
            ),
            'is_cool': is_cool,
            'htc_cool': htc_cool,
            'cp_cool': water_props['cp_water'],
            'rho_cool': water_props['rho_water'],
            'Q_gen': power_generation_data[k],
            'num_cell_seg': num_cell_seg,
            'debug': False,
        }

        try:
            with contextlib.redirect_stdout(io.StringIO()):
                T_dist = (
                    BTMS_model
                    .solve_coolant_temperature_distribution(
                        args,
                        tol=solver_tol,
                        maxiter=solver_maxiter,
                        debug=False,
                    )
                )

        except Exception as e:
            print('\nSolver failed inside run_one_case.')
            print(f'case_id = {case_id}')
            print(
                f'm_dot_total = {m_dot_total} kg/s'
            )
            print(
                f'm_dot_channel = {m_dot_channel} kg/s'
            )
            print(
                f'num_parallel_channels = '
                f'{num_parallel_channels}'
            )
            print(f'T_water_in = {T_water} degC')
            print(f'Dh = {Dh * 1e3} mm')
            print(f'num_cell_seg = {num_cell_seg}')
            print(f'time step k = {k}')
            print(f't = {t:.1f} s')
            print(f'is_cool = {is_cool}')
            print(
                f'Q_gen = {power_generation_data[k]} W'
            )
            print(f'u_water = {u_water}')
            print(f'Re_water = {Re_water}')
            print(f'Nu_water = {Nu_water}')
            print(f'h_water = {h_water}')
            print(f'htc_global = {htc_global}')
            print(f'A_HT_seg = {A_HT_seg}')
            print(f'A_cool_cs = {A_cool_cs}')
            print(f'P_pump = {P_pump}')
            print(
                f'E_pump_cumulative = '
                f'{E_pump_cumulative}'
            )
            print(f'mtotal = {mtotal}')
            print(
                'T_bat_min/max before solve = '
                f'{np.min(T_bat)}, {np.max(T_bat)}'
            )
            print(
                'T_cool_min/max before solve = '
                f'{np.min(T_cool)}, {np.max(T_cool)}'
            )
            print(f'solver_tol = {solver_tol}')
            print(f'solver_maxiter = {solver_maxiter}')
            print(f'error = {repr(e)}')
            raise

        if not np.all(np.isfinite(T_dist)):
            print('\nSolver returned non-finite values.')
            print(f'case_id = {case_id}')
            print(
                f'm_dot_total = {m_dot_total} kg/s'
            )
            print(
                f'm_dot_channel = {m_dot_channel} kg/s'
            )
            print(
                f'num_parallel_channels = '
                f'{num_parallel_channels}'
            )
            print(f'T_water_in = {T_water} degC')
            print(f'Dh = {Dh * 1e3} mm')
            print(f'num_cell_seg = {num_cell_seg}')
            print(f'time step k = {k}')
            print(f't = {t:.1f} s')
            print(
                'T_dist_min/max = '
                f'{np.nanmin(T_dist)}, '
                f'{np.nanmax(T_dist)}'
            )
            raise FloatingPointError(
                'Non-finite values detected in T_dist'
            )

        T_cool = T_dist[:num_seg]
        T_bat = T_dist[num_seg:]

        T_bat_history[k + 1, :] = T_bat
        T_water_history[k + 1, :] = T_cool

    Tmax = np.max(T_bat_history, axis=1)
    Tmin = np.min(T_bat_history, axis=1)
    DeltaT = Tmax - Tmin
    Twater_out = T_water_history[:, -1]

    def idx(t):
        return int(round(t / dt))

    row = {
        'case_id': case_id,

        # Total system mass flow rate: parameter-scan variable.
        'm_dot_total_kg_s': float(m_dot_total),

        # Per-channel mass flow rate: thermal-solver variable.
        'm_dot_channel_kg_s': float(m_dot_channel),

        'T_water_in_C': float(T_water),
        'Dh_mm': float(Dh * 1e3),
        'num_cell_seg': float(num_cell_seg),
        'num_parallel_channels': num_parallel_channels,
        'u_water_m_s': float(u_water),
        'Re_water': float(Re_water),
        'P_pump': P_pump,
        'E_pump_cumulative': E_pump_cumulative,
        'm_plate_kg': float(
            mass_results['m_plate_kg']
        ),
        'm_coolant_kg': float(
            mass_results['m_coolant_kg']
        ),
        'mtotal': mtotal,
        'Tmax_mission_C': float(np.max(Tmax)),
        'DeltaT_mission_max_C': float(
            np.max(DeltaT)
        ),
        'cruise_recovery_Tmax_C': float(
            Tmax[idx(t_cruise_start)]
            - Tmax[idx(t_cruise_end)]
        ),
        'Twater_out_cruise_end_C': float(
            Twater_out[idx(t_cruise_end)]
        ),
    }

    for stage_name, t0, t1 in mission_stages:
        row[f'dTmax_{stage_name}_C'] = float(
            Tmax[idx(t1)] - Tmax[idx(t0)]
        )

    return row

In [16]:
# run all cases and export the final xlsx table

def format_excel_table(xlsx_path):
    """Apply the original Times New Roman table style to the exported Excel file."""
    wb = load_workbook(xlsx_path)
    ws = wb.active
    ws.title = 'Parameter scan'

    body_font = Font(name='Times New Roman', size=10)
    header_font = Font(name='Times New Roman', size=10, bold=True)
    alignment = Alignment(horizontal='center', vertical='center')
    thin = Side(style='thin')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for row in ws.iter_rows():
        for cell in row:
            cell.font = header_font if cell.row == 1 else body_font
            cell.alignment = alignment
            cell.border = border

            if cell.row > 1 and isinstance(cell.value, float):
                cell.number_format = '0.0000'

    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions

    for col_idx, column_cells in enumerate(ws.columns, start=1):
        max_len = max(
            len(str(cell.value))
            if cell.value is not None
            else 0
            for cell in column_cells
        )

        ws.column_dimensions[
            get_column_letter(col_idx)
        ].width = min(max(max_len + 2, 12), 28)

    wb.save(xlsx_path)


summary_rows = []

for n, case in enumerate(scan_cases, start=1):

    num_parallel_channels = int(
        round(N_c / case['num_cell_seg'])
    )

    m_dot_channel = (
        case['m_dot_total']
        / num_parallel_channels
    )

    print(
        f"Running {n}/{len(scan_cases)}: "
        f"{case['case_id']}, "
        f"m_dot_total={case['m_dot_total']} kg/s, "
        f"m_dot_channel={m_dot_channel:.6f} kg/s, "
        f"T_water={case['T_water']} degC, "
        f"num_parallel_channels={num_parallel_channels}, "
        f"num_cell_seg={case['num_cell_seg']}, "
        f"Dh={case['Dh'] * 1e3} mm"
    )

    try:
        summary_rows.append(
            run_one_case(**case)
        )

    except Exception as e:
        print(
            "\nParameter scan stopped because "
            "one case failed."
        )
        print(
            f"failed index = "
            f"{n}/{len(scan_cases)}"
        )
        print(
            f"failed case_id = "
            f"{case['case_id']}"
        )
        print(
            f"failed m_dot_total = "
            f"{case['m_dot_total']} kg/s"
        )
        print(
            f"failed m_dot_channel = "
            f"{m_dot_channel} kg/s"
        )
        print(
            f"failed T_water_in = "
            f"{case['T_water']} degC"
        )
        print(
            f"failed num_parallel_channels = "
            f"{num_parallel_channels}"
        )
        print(
            f"failed num_cell_seg = "
            f"{case['num_cell_seg']}"
        )
        print(
            f"failed Dh = "
            f"{case['Dh'] * 1e3} mm"
        )
        print(f"error = {repr(e)}")

        raise


parameter_scan_table = pd.DataFrame(
    summary_rows
)

column_order = [
    'm_dot_total_kg_s',
    'T_water_in_C',
    'Dh_mm',
    'num_parallel_channels',
    'u_water_m_s',
    'Re_water',
    'P_pump',
    'E_pump_cumulative',
    'm_plate_kg',
    'm_coolant_kg',
    'mtotal',
    'Tmax_mission_C',
    'DeltaT_mission_max_C',
    'cruise_recovery_Tmax_C',
    'Twater_out_cruise_end_C',
    'dTmax_takeoff_C',
    'dTmax_climb_C',
    'dTmax_transition1_C',
    'dTmax_cruise_C',
    'dTmax_transition2_C',
    'dTmax_descent_C',
    'dTmax_hover_C',
    'dTmax_landing_C',
]

parameter_scan_table = (
    parameter_scan_table[column_order]
)

output_dir = OUTPUT_DIR
os.makedirs(
    output_dir,
    exist_ok=True,
)

xlsx_path = os.path.join(
    output_dir,
    RESULT_XLSX_NAME,
)

parameter_scan_table.to_excel(
    xlsx_path,
    index=False,
)

format_excel_table(xlsx_path)

print(
    f'Saved official Excel: {xlsx_path}'
)
print(
    'Only one table file is generated '
    'for this case.'
)

parameter_scan_table.head()



Running 1/480: C0003_S001, m_dot_total=0.045 kg/s, m_dot_channel=0.001607 kg/s, T_water=20.0 degC, num_parallel_channels=28, num_cell_seg=0.5714285714285714, Dh=3.0 mm
Running 2/480: C0003_S002, m_dot_total=0.045 kg/s, m_dot_channel=0.001607 kg/s, T_water=20.0 degC, num_parallel_channels=28, num_cell_seg=0.5714285714285714, Dh=4.0 mm
Running 3/480: C0003_S003, m_dot_total=0.045 kg/s, m_dot_channel=0.001607 kg/s, T_water=20.0 degC, num_parallel_channels=28, num_cell_seg=0.5714285714285714, Dh=5.0 mm
Running 4/480: C0003_S004, m_dot_total=0.045 kg/s, m_dot_channel=0.001607 kg/s, T_water=20.0 degC, num_parallel_channels=28, num_cell_seg=0.5714285714285714, Dh=6.0 mm
Running 5/480: C0003_S005, m_dot_total=0.045 kg/s, m_dot_channel=0.001875 kg/s, T_water=20.0 degC, num_parallel_channels=24, num_cell_seg=0.6666666666666666, Dh=3.0 mm
Running 6/480: C0003_S006, m_dot_total=0.045 kg/s, m_dot_channel=0.001875 kg/s, T_water=20.0 degC, num_parallel_channels=24, num_cell_seg=0.6666666666666666, Dh

,m_dot_total_kg_s,T_water_in_C,Dh_mm,num_parallel_channels,u_water_m_s,Re_water,P_pump,E_pump_cumulative,m_plate_kg,m_coolant_kg,...,cruise_recovery_Tmax_C,Twater_out_cruise_end_C,dTmax_takeoff_C,dTmax_climb_C,dTmax_transition1_C,dTmax_cruise_C,dTmax_transition2_C,dTmax_descent_C,dTmax_hover_C,dTmax_landing_C
0,0.045,20.0,3.0,28,0.227773,681.005631,0.043278,83.094256,2.579877,0.081838,...,1.052304,20.462827,0.412637,1.417092,0.643737,-1.052304,-0.652175,0.526984,10.810119,0.280811
1,0.045,20.0,4.0,28,0.128122,510.754223,0.013694,26.291542,2.806896,0.145490,...,1.055711,20.462682,0.412596,1.416093,0.635834,-1.055711,-0.650525,0.527268,10.781822,0.279220
2,0.045,20.0,5.0,28,0.081998,408.603378,0.005609,10.769016,2.984378,0.227328,...,1.057705,20.462594,0.412571,1.415490,0.631051,-1.057705,-0.649529,0.527440,10.764693,0.278258
3,0.045,20.0,6.0,28,0.056943,340.502815,0.002705,5.193391,3.112323,0.327352,...,1.059014,20.462536,0.412554,1.415086,0.627845,-1.059014,-0.648862,0.527556,10.753211,0.277613
4,0.045,20.0,3.0,24,0.265735,794.506569,0.050491,96.943298,2.611723,0.070147,...,0.996352,20.464852,0.413169,1.429751,0.743456,-0.996352,-0.673327,0.523339,11.166119,0.300911
